[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/vectors/t01_basics.ipynb)

# Vector algebra basics: the one-mark trap

Find the unit vector from P(1, 2, −1) to Q(3, −1, 5). Most of us subtract, write 2i − 3j + 6k and stop. That arrow
points the right way, but its length is 7, and a unit vector has length 1. Divided by 7 it becomes (2/7, −3/7, 6/7).

In this notebook you can redo every problem from the video, and check your own answers.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When a slider appears, drag it and watch. Change any
number and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this cell first.
import math
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # chalk, chalk smudge, ghost lines
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # along an axis · the result · across

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def sub(p, q): return tuple(x - y for x, y in zip(p, q))          # head minus tail: sub(head, tail)
def add(p, q): return tuple(x + y for x, y in zip(p, q))
def times(k, p): return tuple(k * x for x in p)

def length(v):
    """Pythagoras done twice: across the floor, then up."""
    return math.sqrt(sum(x * x for x in v))

def show(v):
    """An arrow's numbers as a column would read them, with fractions kept exact."""
    return "(" + ", ".join(str(Fraction(x).limit_denominator(1000)).replace("-", "−") for x in v) + ")"

def section(c, d, m, n):
    """The point dividing CD internally in the ratio m : n: each end weighted by the other end's number."""
    return tuple(Fraction(n * x + m * y, m + n) for x, y in zip(c, d))

**The drawing tools.** Run the next cell too: it holds the code that draws the figures below. You don't need to read
it to play along.

In [ ]:
def space(ax, lim, view, elev=22):
    """A 3D axes on the slate: no grey panes, a ghost grid, the camera `view` degrees round the vertical."""
    ax.set_facecolor(SLATE)
    for a in (ax.xaxis, ax.yaxis, ax.zaxis):
        a.set_pane_color((0, 0, 0, 0)); a._axinfo["grid"]["color"] = GHOST
    (x0, x1), (y0, y1), (z0, z1) = lim
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_zlim(z0, z1)
    ax.view_init(elev=elev, azim=view)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")

def column(ax, vals, x, y, size=22, col=CHALK, name=""):
    """The arrow's numbers as a column in square brackets, on a flat panel at (x, y) (the top of the column)."""
    rows = [str(Fraction(v).limit_denominator(100)).replace("-", "−") for v in vals]
    gap = 0.12
    for k, s in enumerate(rows):
        ax.text(x, y - k * gap, s, color=col, fontsize=size, ha="center", va="center", family="serif")
    top, bot, w = y + gap * 0.55, y - (len(rows) - 1) * gap - gap * 0.55, 0.11
    for side in (-1, 1):
        bx = x + side * w
        ax.plot([bx, bx], [bot, top], color=DIM, lw=2)
        ax.plot([bx, bx - side * 0.025], [top, top], color=DIM, lw=2)
        ax.plot([bx, bx - side * 0.025], [bot, bot], color=DIM, lw=2)
    if name: ax.text(x - w - 0.04, y - gap * (len(rows) - 1) / 2, name, color=col, fontsize=size, ha="right",
                     va="center", family="serif")

def arrow_figure(tail, head, view, extra=None, lim=((-1, 6), (-4, 4), (-2, 7)), title=""):
    """An arrow in space on the left, its column of numbers beside it on the right: every arrow, two ways."""
    fig = plt.figure(figsize=(11.5, 5))
    ax = fig.add_axes([0.0, 0.0, 0.6, 1.0], projection="3d")
    space(ax, lim, view)
    v = sub(head, tail)
    ax.quiver(*tail, *v, color=CHALK, lw=3, arrow_length_ratio=0.1)
    ax.scatter(*zip(tail, head), color=DIM, s=30, depthshade=False)
    if extra: extra(ax)
    p = fig.add_axes([0.6, 0.0, 0.4, 1.0]); p.axis("off"); p.set_xlim(0, 1); p.set_ylim(0, 1)
    return fig, ax, p, v

P = (1, 2, -1)

def unit_vector(qx=3, qy=-1, qz=5, view=-60):
    Q = (qx, qy, qz)
    fig, ax, p, v = arrow_figure(P, Q, view)
    L = length(v)
    ax.text(*P, "  P", color=DIM); ax.text(*Q, "  Q", color=DIM)
    column(p, v, 0.28, 0.8, name="PQ =")
    p.text(0.62, 0.68, f"length\n√{sum(x * x for x in v)} = {L:.3g}", color=CORAL, fontsize=15, ha="center", va="center")
    if L:
        u = tuple(x / L for x in v)
        column(p, u, 0.28, 0.3, size=18, col=SAGE, name="÷ length =")
        p.text(0.62, 0.18, f"squares add to {sum(x * x for x in u):.3f}", color=SAGE, fontsize=13, ha="center")
    ax.set_title("the arrow PQ: head minus tail")
    plt.show()

## 1 · Every arrow, two ways

Drag the head Q of the arrow from P(1, 2, −1). The column beside it is head minus tail. Its length comes from
Pythagoras done twice, and dividing every number by that length gives the unit vector, the direction alone, whose
squares add to exactly 1. Turn the view with the last slider.

In [ ]:
interact(unit_vector, qx=IntSlider(value=3, min=-1, max=6, description="Q: x"),
         qy=IntSlider(value=-1, min=-4, max=4, description="Q: y"),
         qz=IntSlider(value=5, min=-2, max=7, description="Q: z"),
         view=IntSlider(value=-60, min=-180, max=180, step=10, description="turn"));

## 2 · The method

1. **Ask what the question wants**: a size, a direction, or a place?
2. **Turn every point into an arrow from the origin** (its position vector).
3. **For the arrow between two points, take head minus tail.**
4. **Take the length for a size, or divide by it for a direction.** A place is a weighted average of position vectors.
5. **Check with one number**: a unit vector's squares add to 1.

## 3 · Position vectors, components and equal vectors

The position vector of P(1, 2, −1) is i + 2j − k. The arrow from P to Q is q − p, never p − q. Its scalar components
are 2, −3, 6; its vector components are 2i, −3j, 6k. Equal vectors match number for number.

In [ ]:
Q = (3, -1, 5)
print("position vector of P:", show(P), "  of Q:", show(Q))
print("PQ = q − p =", show(sub(Q, P)), "   (p − q =", show(sub(P, Q)), "points the other way)")
print("scalar components:", ", ".join(str(x) for x in sub(Q, P)),
      "   vector components:", "  ".join(f"{x}{e}" for x, e in zip(sub(Q, P), ("i", "j", "k"))))
# equal vectors: (x + y, 3, −1) = (5, x − y, −1), so x + y = 5 and x − y = 3
x = Fraction(5 + 3, 2); y = 5 - x
print(f"equal vectors: x + y = 5, x − y = 3  →  x = {x}, y = {y};  both read", show((x + y, 3, -1)), "=", show((5, x - y, -1)))

## 4 · Sizes: magnitude and the distance formula

|a| = √(a₁² + a₂² + a₃²). The distance between two points is the length of the arrow between them. Which point on the
y axis is equally far from (3, 1, −2) and (−1, 5, 2)? Any point there is (0, y, 0).

In [ ]:
v = (2, -3, 6)
print(f"floor: 2² + 3² = {2 ** 2 + 3 ** 2},  then up: 13 + 6² = {13 + 36},  length = {length(v):g}")
A, B = (3, 1, -2), (-1, 5, 2)
for y in range(-1, 5):
    da, db = sum(t * t for t in sub((0, y, 0), A)), sum(t * t for t in sub((0, y, 0), B))
    print(f"(0, {y}, 0):  squared distances {da:3d} and {db:3d}" + ("   ← equal" if da == db else ""))
print("both distances at (0, 2, 0): √14 =", round(math.sqrt(14), 3))

## 5 · Directions: direction cosines and direction ratios

Divide by the length: 2i − j + 2k has length 3, so its direction cosines are l, m, n = 2/3, −1/3, 2/3. Ratios are
any three numbers in the same proportion. Only the cosines square-add to 1. And a drone making 60° with x and 45°
with y: cos²γ = 1 − 1/4 − 1/2.

In [ ]:
d = (2, -1, 2)
lmn = tuple(Fraction(t, int(length(d))) for t in d)
print("direction cosines:", show(lmn), "  squares add to", sum(t * t for t in lmn))
print("direction ratios: (2, −1, 2) or", show(times(2, d)), "  their squares add to", sum(t * t for t in d), "(not 1)")
n2 = 1 - Fraction(1, 4) - Fraction(1, 2)
print(f"drone: n² = {n2}, n = ±1/2, γ = {math.degrees(math.acos(0.5)):.0f}° or {math.degrees(math.acos(-0.5)):.0f}° (it climbs: 60°)")
print(f"45° with all three axes? 3 × cos²45° = {3 * math.cos(math.radians(45)) ** 2:.2f}, not 1: no such vector")

## 6 · Parallel arrows and collinear points

Two arrows are parallel when one is a number times the other. Three points are on one line when the arrows from one
of them are parallel.

In [ ]:
print("(2, −3, p) ∥ (−4, 6, 8): the factor is", Fraction(-4, 2), " so −2p = 8 and p =", Fraction(8, -2))
K, L, M = (1, -1, 2), (3, 2, 1), (7, 8, -1)
KL, KM = sub(L, K), sub(M, K)
print("KL =", show(KL), "  KM =", show(KM), "  KM = 3 KL:", KM == times(3, KL))

## 7 · Places: the section formula, outside, and the centroid

r = (n c + m d) ÷ (m + n): each end is weighted by the other end's number. Externally one weight turns negative:
r = (m d − n c) ÷ (m − n). The centroid is the average of the three corners.

In [ ]:
C, D = (2, -1, 4), (5, 8, -2)
print("CD =", show(sub(D, C)), "  a third of it:", show(times(Fraction(1, 3), sub(D, C))))
print("R dividing CD in 1 : 2, inside:", show(section(C, D, 1, 2)))
out = tuple(Fraction(1 * dd - 2 * cc, 1 - 2) for cc, dd in zip(C, D))
print("R dividing CD in 1 : 2, outside:", show(out), "  check RD = 2 RC:", sub(D, out) == times(2, sub(C, out)))
U, V, G = (2, -3, 4), (-1, 5, 1), (1, 2, 2)
W = sub(sub(times(3, G), U), V)
print("centroid: W = 3g − u − v =", show(W), "  average of U, V, W:", show(times(Fraction(1, 3), add(add(U, V), W))))

## 8 · The case study: a camera over a stadium

One corner of the pitch is the origin, x and y run along two edges, z straight up, in metres. The camera glides from
S(2, 1, 10) to T(8, 7, 13). Each part wants a size, a direction or a place. Turn the view with the slider.

In [ ]:
S, T = (2, 1, 10), (8, 7, 13)
ST = sub(T, S)
print("(i) arrow:", show(ST), "  (ii) size:", length(ST), "m   (iii) direction:", show(times(Fraction(1, 9), ST)),
      "  (iv) place, 1 : 2:", show(section(S, T, 1, 2)))

def stadium(view=-55):
    def pitch(ax):
        for k in range(0, 11, 2):
            ax.plot([k, k], [0, 8], [0, 0], color=GHOST, lw=1); ax.plot([0, 10], [k * 0.8, k * 0.8], [0, 0], color=GHOST, lw=1)
        ax.plot(*zip(S, T), ":", color=DIM, lw=1.5)
        R = section(S, T, 1, 2)
        ax.scatter(*[float(t) for t in R], color=CORAL, s=90, depthshade=False)
        ax.text(*[float(t) for t in R], "  pause", color=CORAL)
        ax.text(*S, "  S", color=DIM); ax.text(*T, "  T", color=DIM)
    fig, ax, p, v = arrow_figure(S, T, view, pitch, ((0, 10), (0, 8), (0, 14)))
    column(p, v, 0.3, 0.75, name="ST =")
    p.text(0.65, 0.63, f"size: {length(v):g} m", color=CHALK, fontsize=15, ha="center")
    p.text(0.5, 0.3, "direction: " + show(times(Fraction(1, 9), v)), color=SAGE, fontsize=14, ha="center")
    p.text(0.5, 0.18, "pauses at " + show(section(S, T, 1, 2)), color=CORAL, fontsize=14, ha="center")
    plt.show()

interact(stadium, view=IntSlider(value=-55, min=-180, max=180, step=10, description="turn"));

## 9 · The working that loses a mark

M(−2, 3, 5), N(4, 0, −1), divided internally in the ratio 2 : 1. Swapping the weights gives a point only a third of
the way from M.

In [ ]:
Mp, Np = (-2, 3, 5), (4, 0, -1)
print("swapped weights, (2m + n) ÷ 3:", show(tuple(Fraction(2 * a + b, 3) for a, b in zip(Mp, Np))), "  a third of the way")
print("right, (m + 2n) ÷ 3:          ", show(section(Mp, Np, 2, 1)), "  two thirds of the way")
print("walk it: m + (2/3) MN =", show(add(Mp, times(Fraction(2, 3), sub(Np, Mp)))))

## 10 · Check your answer

Three checkers. Type each point or arrow as three numbers separated by commas (fractions like `2/7` are fine), then
press **check**. Run the first cell below once (it sets up the checkers), then the ones after it.

In [ ]:
from IPython.display import display
from ipywidgets import Button, Output, HBox, VBox, Layout, Text

def read(s):
    s = s.replace("−", "-").replace("(", "").replace(")", "")
    return tuple(Fraction(t.strip()) for t in s.split(",") if t.strip())

def check_unit(p, q, yours):
    """Is your answer the unit vector from P to Q?"""
    try:
        p, q, y = read(p), read(q), read(yours)
    except Exception:
        print("I can't read that yet: type three numbers with commas, like 1, 2, -1"); return
    v = sub(q, p)
    L = length(v)
    size = math.sqrt(sum(t * t for t in y))
    same_way = L > 0 and size > 0 and all(abs(a / L - b / size) < 1e-9 for a, b in zip(v, y))
    if not same_way:
        print(f"✗ not the direction of PQ: head minus tail gives PQ = {show(v)}"); return
    if abs(size - 1) < 1e-9:
        print(f"✓ right: a unit vector from P to Q (its squares add to 1)"); return
    print(f"✗ length {size:g}, not a unit vector: it points the right way, so divide every number by {size:g}")
    print(f"   the unit vector is {show(tuple(Fraction(t) / Fraction(size).limit_denominator(1000) for t in y))}")

def check_section(c, d, ratio, yours):
    """Is your point the one dividing CD internally in the ratio m : n?"""
    try:
        c, d, y = read(c), read(d), read(yours)
        m, n = (Fraction(t) for t in ratio.replace(":", ",").split(","))
    except Exception:
        print("I can't read that yet: points as three numbers with commas, the ratio like 1 : 2"); return
    r = section(c, d, m, n)
    if y == r: print(f"✓ right: {show(r)} is {m}/{m + n} of the way from C to D"); return
    print(f"✗ not quite: the point is {show(r)}, {m}/{m + n} of the way from C (each end takes the other end's number)")
    swapped = section(c, d, n, m)
    if y == swapped: print("   yours has the weights swapped")

def check_cosines(vector, yours):
    """Are your three numbers the direction cosines of the vector?"""
    try:
        v, y = read(vector), read(yours)
    except Exception:
        print("I can't read that yet: type three numbers with commas"); return
    L = length(v)
    right = all(abs(float(a) / L - float(b)) < 1e-9 for a, b in zip(v, y))
    if right: print("✓ right: the direction cosines (their squares add to 1)"); return
    print(f"✗ not the cosines: theirs is {show(tuple(t / L for t in v)) if L == int(L) else tuple(round(t / L, 4) for t in v)}")
    if sum(t * t for t in y) != 1: print("   yours don't square-add to 1: ratios, perhaps? Divide by the length.")

def checker(fn, values, labels):
    """Boxes, a check button, and the verdict under them."""
    boxes = [Text(value=v, description=n, layout=Layout(width="230px")) for v, n in zip(values, labels)]
    check_button, answer = Button(description="check"), Output()
    def on_check(_=None):
        answer.clear_output()
        with answer: fn(*(b.value for b in boxes))
    check_button.on_click(on_check)
    display(VBox([HBox(boxes + [check_button]), answer]))
    on_check()

**A unit vector from P to Q.** Type P, Q and your answer.

In [ ]:
checker(check_unit, ["1, 2, -1", "3, -1, 5", "2/7, -3/7, 6/7"], ["P =", "Q =", "yours:"])

**A dividing point.** Type C, D, the ratio m : n and your point.

In [ ]:
checker(check_section, ["2, -1, 4", "5, 8, -2", "1 : 2", "3, 2, 2"], ["C =", "D =", "m : n =", "yours:"])

**Direction cosines.** Type the vector and your l, m, n.

In [ ]:
checker(check_cosines, ["2, -1, 2", "2/3, -1/3, 2/3"], ["vector:", "l, m, n:"])

## 11 · Slide the dividing point

C(2, −1, 4) and D(5, 8, −2). Drag the ratio: the dividing point slides from C to D, and its column (its position
vector, n c + m d over m + n) updates as it goes.

In [ ]:
C, D = (2, -1, 4), (5, 8, -2)

def dividing(m=2, view=-60):
    n = 6 - m
    R = section(C, D, m, n)
    def seg(ax):
        ax.plot(*zip(C, D), color=DIM, lw=2)
        ax.scatter(*[float(t) for t in R], color=CORAL, s=110, depthshade=False)
        ax.text(*C, "  C", color=DIM); ax.text(*D, "  D", color=DIM)
        ax.text(*[float(t) for t in R], "  R", color=CORAL)
    fig, ax, p, v = arrow_figure((0, 0, 0), tuple(float(t) for t in R), view, seg, ((-1, 6), (-2, 9), (-3, 5)))
    column(p, R, 0.3, 0.75, col=CORAL, name="r =")
    g = math.gcd(m, n) or 1
    p.text(0.55, 0.32, f"ratio m : n = {m} : {n}" + (f"  =  {m // g} : {n // g}" if g > 1 else ""), color=CHALK,
           fontsize=15, ha="center")
    where = "at C" if m == 0 else ("at D" if m == 6 else f"{Fraction(m, 6)} of the way from C to D")
    p.text(0.55, 0.2, where, color=DIM, fontsize=13, ha="center")
    ax.set_title("the position vector of the dividing point")
    plt.show()

interact(dividing, m=IntSlider(value=2, min=0, max=6, description="ratio m"),
         view=IntSlider(value=-60, min=-180, max=180, step=10, description="turn"));

**Try it.** The challenge from the video: a vector of length 12 has direction cosines l, m, n, all positive, with
l = m = 2n. What is the vector? Hint: l² + m² + n² = 1. Check your l, m, n with the cosines checker above.

## Sources
NCERT Mathematics Class XII, Part II, ch. 10 (Vector Algebra) and Class XI, ch. 11 (Introduction to Three
Dimensional Geometry); CBSE Mathematics syllabus 2026–27. All retrieved 2026-10-26. Every problem is our own.